# Week 5 Lab: Code Efficiency and the Central Limit Theorem

## Learning Objectives

By the end of this lab, you will be able to:
1. Identify and fix performance bottlenecks in numerical code
2. Use Julia's `@time` macro to benchmark code
3. Simulate and visualize the Central Limit Theorem
4. Understand why the CLT matters for statistical inference

## Why This Matters

**Efficient code:** Econometric simulations (bootstrap, Monte Carlo, MCMC) often require millions of iterations. A 10x speedup can mean the difference between waiting 10 minutes versus 100 minutes.

**The CLT:** The Central Limit Theorem is the foundation of classical inference. It explains why t-statistics and confidence intervals work, even when the underlying data is non-normal.

---

*This lab continues from Week 4. We'll improve the WLLN code from last week, then move on to the CLT.*

## Part I: Improving Code Efficiency

Last week, we wrote code to visualize the WLLN. The code worked, but it was slow. Today we'll identify why and fix it.

In [ ]:
using Distributions
using Random
using Plots

## Exercise 1: Time Your Code

Here's our code from last week. We'll use the `@time` macro to measure execution time.

**Important:** The first run includes compilation time. Run the cell twice—the second run gives the true execution time.

In [ ]:
# Original WLLN simulation code
# Start with n_max = 10³ to keep timing manageable

@time begin
    n_max = 10^3
    μ = 2.0
    
    d = Exponential(μ)
    Random.seed!(42)
    super_sample = rand(d, n_max)
    
    ȳ = Vector{Float64}(undef, n_max)
    for n in 1:n_max
        sub_sample = super_sample[1:n]  # Creates a NEW array each iteration!
        ȳ[n] = mean(sub_sample)
    end
    
    plot(1:n_max, ȳ, label = "Sample mean", xlabel = "Sample size n")
    hline!([μ], linewidth = 2, label = "μ = $μ")
end

In [ ]:
# Now try n_max = 10⁵
# Warning: this may be slow!

@time begin
    n_max = 10^5
    μ = 2.0
    
    d = Exponential(μ)
    Random.seed!(42)
    super_sample = rand(d, n_max)
    
    ȳ = Vector{Float64}(undef, n_max)
    for n in 1:n_max
        sub_sample = super_sample[1:n]  # O(n) operation inside O(n) loop = O(n²)!
        ȳ[n] = mean(sub_sample)
    end
    
    plot(1:n_max, ȳ, label = "Sample mean", xlabel = "Sample size n")
    hline!([μ], linewidth = 2, label = "μ = $μ")
end

**Why is this slow?**

The line `sub_sample = super_sample[1:n]` creates a *new* array of size $n$ at each iteration. This means:
- Iteration 1: create array of size 1, compute mean
- Iteration 2: create array of size 2, compute mean
- ...
- Iteration $N$: create array of size $N$, compute mean

Total work: $1 + 2 + \cdots + N = N (N+1) /2 = O(N^2)$. This is called **quadratic time complexity**.

## Exercise 2: Wrap Code in a Function

First, let's wrap our slow code in a function. This makes it easier to compare different approaches.

**Julia tip:** Wrapping code in a function often improves performance because Julia's JIT compiler can optimize function code better than top-level code.

In [ ]:
"""
    wlln_sim_1(n_max, μ; seed=42)

Simulate the Weak Law of Large Numbers (naive implementation).

This function uses a straightforward but *inefficient* approach that
recomputes the mean from scratch at each step. Complexity: O(n²).

# Arguments
- `n_max::Integer`: Maximum sample size
- `μ::Real`: Parameter of the exponential distribution (also the mean)
- `seed::Integer`: Random seed for reproducibility

# Returns
A plot showing the convergence of the sample mean to μ.
"""
function wlln_sim_1(n_max::Integer, μ::Real; seed::Integer = 42)
    d = Exponential(μ)
    Random.seed!(seed)
    super_sample = rand(d, n_max)
    
    ȳ = Vector{Float64}(undef, n_max)
    for n in 1:n_max
        sub_sample = super_sample[1:n]
        ȳ[n] = mean(sub_sample)
    end
    
    p = plot(1:n_max, ȳ, label = "Sample mean", xlabel = "Sample size n")
    hline!([μ], linewidth = 2, label = "μ = $μ")
    return p
end

In [ ]:
# Test the function
@time wlln_sim_1(1000, 2.0)

In [ ]:
# Time with larger n_max
@time wlln_sim_1(10^5, 2.0)

## Exercise 3: Make Your Code More Efficient

Instead of recalculating the mean from scratch each time, we can use **cumulative sums**.

**Key insight:** The running mean at position $n$ is:

$$\bar{Y}_n = \frac{1}{n}\sum_{i=1}^n Y_i = \frac{S_n}{n}$$

where $S_n = \sum_{i=1}^n Y_i$ is the cumulative sum. Julia's `cumsum()` function computes all cumulative sums in $O(n)$ time.

**Complexity improvement:** From $O(n^2)$ to $O(n)$—a dramatic speedup for large $n$!

**Your Task:** Implement `wlln_sim_2` using `cumsum()` instead of the loop.

In [ ]:
"""
    wlln_sim_2(n_max, μ; seed=42)

Simulate the Weak Law of Large Numbers (efficient implementation).

Uses cumulative sums to compute running means in O(n) time instead of O(n²).

# Arguments
- `n_max::Integer`: Maximum sample size
- `μ::Real`: Parameter of the exponential distribution
- `seed::Integer`: Random seed for reproducibility

# Returns
A plot showing the convergence of the sample mean to μ.
"""
function wlln_sim_2(n_max::Integer, μ::Real; seed::Integer = 42)
    d = Exponential(μ)
    Random.seed!(seed)
    super_sample = rand(d, n_max)
    
    # Efficient: compute all running means at once
    # Hint: cumsum gives [Y₁, Y₁+Y₂, Y₁+Y₂+Y₃, ...]
    # Dividing by [1, 2, 3, ...] gives [ȳ₁, ȳ₂, ȳ₃, ...]
    # Use broadcasting: cumsum(super_sample) ./ (1:n_max)
    
    ȳ = nothing  # Replace with cumsum-based computation
    
    # YOUR CODE HERE
    error("Not yet implemented")
    
    p = plot(1:n_max, ȳ, label = "Sample mean", xlabel = "Sample size n")
    hline!([μ], linewidth = 2, label = "μ = $μ")
    return p
end

In [ ]:
# Compare timing: should be MUCH faster
# Uncomment when ready:

# @time wlln_sim_2(1000, 2.0)

In [ ]:
# Now n_max = 10⁵ should be nearly instant
# Uncomment when ready:

# @time wlln_sim_2(10^5, 2.0)

### Tradeoffs: Readability vs. Speed

**`wlln_sim_1` (slow):**
- Pro: The loop mirrors the mathematical definition of running means
- Con: $O(n^2)$ complexity makes it impractical for large simulations

**`wlln_sim_2` (fast):**
- Pro: $O(n)$ complexity—handles millions of observations easily
- Con: The `cumsum(...) ./ (...)` idiom requires understanding vectorized operations

**Lesson:** Write clear code first, then optimize if needed. Use comments to explain clever optimizations.

---

# Part II: The Central Limit Theorem

The WLLN tells us *where* the sample mean converges. The **Central Limit Theorem (CLT)** tells us about the *distribution* of the sample mean before it fully converges.

## Statement of the CLT

For a random sample $Y_1, \ldots, Y_N$ with $E(Y_i) = \mu$ and $\text{Var}(Y_i) = \sigma^2$:

$$\frac{\bar{Y}_N - \mu}{\sigma / \sqrt{N}} \xrightarrow{d} N(0, 1) \quad \text{as } N \to \infty$$

Equivalently, for large $N$:

$$\bar{Y}_N \overset{\text{approx}}{\sim} N\left(\mu, \frac{\sigma^2}{N}\right)$$

## Why the CLT Matters for Econometrics

1. **Inference without normality:** We can construct confidence intervals and test hypotheses even when the underlying data is non-normal (e.g., wages, firm sizes, durations).

2. **OLS asymptotics:** The sampling distribution of $\hat{\beta}$ is approximately normal for large samples, which justifies using t-tests and F-tests.

3. **The "$N = 30$ rule":** A common heuristic says the CLT "kicks in" around $N = 30$, but this depends heavily on how non-normal the underlying distribution is.

## Exercise 4: Visualize the CLT

We'll use exponentially distributed data with $\mu = 2$ (so $\sigma^2 = 4$).

According to the CLT:
$$\bar{Y}_N \overset{\text{approx}}{\sim} N\left(2, \frac{4}{N}\right)$$

**Simulation approach:**
1. Fix sample size $N$
2. Draw many independent samples of size $N$
3. Compute $\bar{Y}$ for each sample
4. Plot histogram of sample means and compare to normal density

### Worked Example: CLT with N = 1

First, let's see what happens with no averaging ($N = 1$):

In [ ]:
# CLT simulation: N = 1 (no averaging, just the raw exponential distribution)

μ = 2.0
N = 1       # Sample size
rep = 1000  # Number of independent samples

Random.seed!(42)
d = Exponential(μ)

# Store the mean of each sample
ȳ = Vector{Float64}(undef, rep)

for r in 1:rep
    random_sample = rand(d, N)      # Draw sample of size N
    ȳ[r] = mean(random_sample)      # Compute and store mean
end

# Plot histogram with normal approximation overlay
histogram(ȳ, normalize = true, alpha = 0.7,
          label = "Sample means (N = $N)",
          xlabel = "ȳ",
          ylabel = "Density",
          title = "CLT: Distribution of ȳ (N = $N)")

# Overlay the CLT-implied normal distribution
σ_ȳ = μ / sqrt(N)  # Standard deviation of ȳ under CLT
plot!(x -> pdf(Normal(μ, σ_ȳ), x), 0, 12,
      linewidth = 3, color = :red,
      label = "N($μ, $(μ^2)/$N)")

**Observation:** With $N = 1$, there's no averaging—$\bar{Y}_1 = Y_1$ is just the raw exponential distribution. The normal approximation is terrible! The exponential is right-skewed, while the normal is symmetric.

### Your Turn: Compare Different Sample Sizes

Create a grid of plots showing how the CLT approximation improves as $N$ increases.

**Hints:**
- Use a `for` loop over `N in [1, 5, 10, 30, 100, 1000]`
- Store each plot in a vector using `push!(plots, p)`
- Combine with `plot(plots..., layout = (2, 3), size = (900, 600))`

In [ ]:
# Compare different sample sizes to see CLT convergence

plots = []  # Will store our plots

for N in [1, 5, 10, 30, 100, 1000]
    μ = 2.0
    rep = 1000
    
    Random.seed!(42)
    d = Exponential(μ)
    
    ȳ = nothing  # Vector to store sample means
    
    # YOUR CODE HERE
    # 1. Create a vector ȳ of size rep
    # 2. Loop r in 1:rep, draw a sample of size N, compute mean, store in ȳ[r]
    # 3. Compute σ_ȳ = μ / sqrt(N)
    # 4. Create histogram with normal overlay
    # 5. push!(plots, p)
    
    error("Not yet implemented")
end

# plot(plots..., layout = (2, 3), size = (900, 600))

**Key observations:**

1. **N = 1:** The histogram is the exponential distribution itself—highly skewed.

2. **N = 5:** Still noticeably right-skewed, but less extreme.

3. **N = 10:** Starting to look more symmetric.

4. **N = 30:** The normal approximation is quite good (the "rule of 30").

5. **N = 100, 1000:** Almost indistinguishable from normal. Note also the narrowing of the distribution as $N$ increases (variance $= \mu^2/N$ shrinks).

## Exercise 5: Create a CLT Simulation Function

Let's wrap our simulation in a reusable function.

**Task:** Complete the `clt_sim` function that draws `rep` independent samples of size `N`, computes the sample mean of each, and plots the histogram with CLT-implied normal overlay.

In [ ]:
"""
    clt_sim(μ, N, rep; seed=42)

Simulate the Central Limit Theorem for an exponential distribution.

Draws `rep` independent samples of size `N` from Exp(μ), computes the 
sample mean of each, and plots the histogram with the CLT-implied 
normal density overlay.

# Arguments
- `μ::Real`: Parameter of the exponential distribution (= mean = √variance)
- `N::Integer`: Sample size for each draw
- `rep::Integer`: Number of independent samples (replications)
- `seed::Integer`: Random seed for reproducibility

# Returns
A plot comparing the empirical distribution of ȳ to the CLT approximation.

# Theoretical background
For Y ~ Exp(μ), we have E(Y) = μ and Var(Y) = μ².
The CLT implies ȳ ~ N(μ, μ²/N) approximately for large N.
"""
function clt_sim(μ::Real, N::Integer, rep::Integer; seed::Integer = 42)
    Random.seed!(seed)
    d = Exponential(μ)
    
    ȳ = nothing  # Vector to store sample means
    
    # YOUR CODE HERE
    # 1. Create vector ȳ of size rep
    # 2. Loop r in 1:rep:
    #      random_sample = rand(d, N)
    #      ȳ[r] = mean(random_sample)
    # 3. Compute σ_ȳ = μ / sqrt(N)
    # 4. Create histogram and normal overlay
    
    error("Not yet implemented")
    
    # CLT-implied standard deviation of sample mean
    σ_ȳ = μ / sqrt(N)
    
    p = histogram(ȳ, normalize = true, alpha = 0.7,
                  label = "Empirical distribution of ȳ",
                  xlabel = "Sample mean ȳ",
                  ylabel = "Density",
                  title = "CLT Simulation (μ=$μ, N=$N)")
    
    plot!(p, x -> pdf(Normal(μ, σ_ȳ), x),
          color = :red, linewidth = 3,
          label = "N($μ, $(round(μ^2/N, digits=4)))")
    
    return p
end

In [ ]:
# Test: μ = 2, N = 100, 1000 replications
# Uncomment when ready:

# clt_sim(2.0, 100, 1000)

In [ ]:
# With very large N and many replications, the CLT is almost exact
# Uncomment when ready:

# clt_sim(2.0, 10_000, 10_000)

**Interpretation:** With $N = 10,000$ observations per sample, the distribution of $\bar{Y}$ is almost perfectly normal. The variance is tiny ($\mu^2/N = 4/10000 = 0.0004$), so the sample mean is very precisely estimated.

## Summary

### Part I: Code Efficiency

- **Profile before optimizing:** Use `@time` to identify actual bottlenecks.
- **Avoid $O(n^2)$ patterns:** Recalculating from scratch inside a loop is a common trap.
- **Vectorize when possible:** `cumsum` replaced a loop with a single, efficient operation.

### Part II: The Central Limit Theorem

- **The CLT applies to averages:** Even if individual observations are highly skewed, their average is approximately normal.
- **Rate of convergence varies:** More skewed distributions need larger $N$ before the normal approximation works well.
- **Practical guidance:** For moderately non-normal data, $N \approx 30$ often suffices. For highly skewed data, larger samples may be needed.

**Next week:** We'll apply these concepts to understand the sampling distribution of OLS estimators.